# Figure 5d,e — Sensory input composition of PSI-IhN postsynaptic neurons

| Panel | Content |
| --- | --- |
| **d** | One row per postsynaptic spinal cord neuron, grouped by the PSI-IhN population that contacts it; columns = sensory subtype; colour = that subtype's fraction of the neuron's sensory input |
| **e** | The same composition pooled per population, as stacked bars |

Together these show that neurons receiving feedforward inhibition from a PSI-IhN often
sample *different* afferents than the PSI-IhN itself — the heterotypic motif quantified in
panel c.

**Input.** `../data/psi_ihn_postsynaptic_feather/` and
`../data/psi_ihn_partners_filtered_gt400.csv`. **Runs offline — no CAVE access.**


In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "svg.fonttype": "none",
    "font.size": 6, "axes.titlesize": 6, "axes.labelsize": 6,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
})

CM = 1 / 2.54


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")
FEATHER_DIR = DATA / "psi_ihn_postsynaptic_feather"
PARTNERS_CSV = DATA / "psi_ihn_partners_filtered_gt400.csv"

# Only neurons postsynaptic to PSI-IhN *axons* (the axodendritic FFI motif).
KIND = "axon_pre"

SENSORY_TYPES = ["Aδ-HTMR", "C-Cold", "C-HTMR", "C-HTMR-Non-peptidergic",
                 "C-Heat", "Aβ-LTMR", "Aδ-LTMR", "C-LTMR"]
LOW_CONF_LABEL = "low_confidence_sensory"
CLASSIFIER_COLUMNS = ["cltmr", "abltmr", "adltmr", "chtmrnp",
                      "c-htmr", "adhtmrp", "cheat", "cold"]
LOW_CONF_THRESHOLD = 0.6

# Every sensory category that counts toward a neuron's denominator ("N/A" = non-sensory).
COUNTED_TYPES = SENSORY_TYPES + [LOW_CONF_LABEL]

TAGS = ["ad-htmr", "islet-chtmr", "islet-cltmr", "islet-adltmr", "IN-abltmr"]

TAG_LABELS = {
    "ad-htmr":      "PSI-IhN Aδ-HTMR → postsyn.",
    "islet-chtmr":  "Islet-PSI-IhN C-HTMR/Heat → postsyn.",
    "islet-cltmr":  "Islet-PSI-IhN C-LTMR → postsyn.",
    "islet-adltmr": "Islet-PSI-IhN Aδ-LTMR → postsyn.",
    "IN-abltmr":    "PSI-IhN Aβ-LTMR → postsyn.",
}

# The one sensory subtype each PSI-IhN population is itself driven by. The
# homotypic selectivity index is the fraction of a postsynaptic neuron's sensory
# input that comes from this subtype.
TAG_TARGET_TYPE = {
    "ad-htmr":      "Aδ-HTMR",
    "islet-chtmr":  "C-HTMR-Non-peptidergic",
    "islet-cltmr":  "C-LTMR",
    "islet-adltmr": "Aδ-LTMR",
    "IN-abltmr":    "Aβ-LTMR",
}

TAG_COLOURS = {
    "ad-htmr":      "#F28585",
    "islet-chtmr":  "#3C2F80",
    "islet-cltmr":  "#789342",
    "islet-adltmr": "#9AEBA3",
    "IN-abltmr":    "#45C4B0",
}

DISPLAY = {
    "C-Heat": "Peptidergic C-Nociceptors",
    "C-HTMR": "C-HTMR/Heat (SST)",
    "C-HTMR-Non-peptidergic": "C-HTMR/Heat (MrgprA3,MrgprD,MrgprB4)",
    LOW_CONF_LABEL: "Low-Confidence",
}

SENSORY_COLOURS = {
    "C-Cold":                               "#13678A",
    "Aδ-HTMR":                              "#F0AE44",
    "Peptidergic C-Nociceptors":            "#AE7F21",
    "C-HTMR/Heat (SST)":                    "#ffbbff",
    "C-HTMR/Heat (MrgprA3,MrgprD,MrgprB4)": "#548D95",
    "C-LTMR":                               "#789342",
    "Aδ-LTMR":                              "#9AEBA3",
    "Aβ-LTMR":                              "#45C4B0",
    "Low-Confidence":                       "#d3d3d3",
}


In [ ]:
def sensory_counts_for_neuron(post_root_id):
    """Sensory synapse counts for one postsynaptic neuron, or None if unusable.

    A synapse SPINE calls sensory but whose every subtype probability is
    <= LOW_CONF_THRESHOLD is relabelled `low_confidence_sensory` — it stays in the
    denominator as its own category rather than being dropped or forced elsewhere.
    """
    path = FEATHER_DIR / f"cell_type_predictions_{int(post_root_id)}.feather"
    if not path.exists():
        return None
    try:
        f = pd.read_feather(path)
    except Exception:
        return None
    if "cell_type" not in f.columns or "is_sensory" not in f.columns:
        return None
    if any(c not in f.columns for c in CLASSIFIER_COLUMNS):
        return None

    low_conf = (f[CLASSIFIER_COLUMNS] <= LOW_CONF_THRESHOLD).all(axis=1)
    f.loc[(f["is_sensory"] == True) & low_conf, "cell_type"] = LOW_CONF_LABEL

    vc = f["cell_type"].astype(str).value_counts().to_dict()
    counts = {ct: int(vc.get(ct, 0)) for ct in COUNTED_TYPES}
    if sum(counts.values()) <= 0:
        return None          # readable but nothing classifiable — skip, don't keep as zeros
    return counts


def build_postsynaptic_table():
    """One row per (PSI-IhN population, postsynaptic neuron) with its sensory composition.

    A postsynaptic neuron contacted by more than one population is assigned to the
    population supplying the most synapses, so each neuron appears once per figure.
    """
    partners = pd.read_csv(PARTNERS_CSV)
    partners = partners[partners["kind"] == KIND]
    partners = partners[partners["tag"].isin(TAGS)]

    # Collapse repeated (tag, post) edges. A neuron contacted by more than one PSI-IhN
    # population stays in EACH of them -- per the Methods, "a single spinal cord neuron
    # could belong to multiple postsynaptic neuron groups". So the unit here is the
    # PSI-IhN -> postsynaptic-neuron *connection*, not the neuron.
    edges = (partners.groupby(["tag", "post_root_id"], as_index=False)["n_synapses"]
                     .sum())

    rows, skipped = [], 0
    for _, e in edges.iterrows():
        counts = sensory_counts_for_neuron(e.post_root_id)
        if counts is None:
            skipped += 1
            continue
        denom = sum(counts.values())
        row = {"tag": e.tag, "post_root_id": int(e.post_root_id),
               "n_synapses_from_tag": int(e.n_synapses), "n_sensory": denom}
        row.update(counts)
        row["target_frac"] = counts[TAG_TARGET_TYPE[e.tag]] / denom
        rows.append(row)

    table = pd.DataFrame(rows)
    print(f"{len(edges)} PSI-IhN -> postsynaptic connections "
          f"({edges.post_root_id.nunique()} distinct neurons); "
          f"{len(table)} usable, {skipped} skipped "
          f"(feather missing or no classifiable sensory synapses)")
    print(table.tag.value_counts().reindex(TAGS).to_string())
    return table


postsyn = build_postsynaptic_table()


In [ ]:
# Column order used in the published panels.
COLUMN_ORDER = ["C-Cold", "Aδ-HTMR", "C-Heat", "C-HTMR", "C-HTMR-Non-peptidergic",
                "C-LTMR", "Aδ-LTMR", "Aβ-LTMR", LOW_CONF_LABEL]

frac = postsyn[COLUMN_ORDER].div(postsyn[COLUMN_ORDER].sum(axis=1), axis=0).fillna(0)
frac.columns = [DISPLAY.get(c, c) for c in COLUMN_ORDER]
frac["tag"] = postsyn["tag"].to_numpy()

# Within each population, sort neurons by their dominant sensory subtype then by how
# strongly it dominates, so the blocks in the heatmap read cleanly.
value_cols = [c for c in frac.columns if c != "tag"]
frac["dominant"] = frac[value_cols].idxmax(axis=1)
frac["dominant_frac"] = frac[value_cols].max(axis=1)
frac["tag"] = pd.Categorical(frac["tag"], categories=TAGS, ordered=True)
frac["dominant"] = pd.Categorical(frac["dominant"], categories=value_cols, ordered=True)
ordered = frac.sort_values(["tag", "dominant", "dominant_frac"],
                           ascending=[True, True, False])

sizes = ordered.groupby("tag", observed=True).size().reindex(TAGS).fillna(0).astype(int)
print(sizes.to_string())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig = plt.figure(figsize=(7 * CM, 9 * CM))
grid = fig.add_gridspec(1, 2, width_ratios=[3.2, 10], wspace=0.0)

ax_lab = fig.add_subplot(grid[0, 0])
ax_lab.set_xlim(0, 1)
ax_lab.set_ylim(0, len(ordered))
ax_lab.invert_yaxis()
ax_lab.axis("off")
start = 0
for tag in TAGS:
    n = sizes[tag]
    if n:
        ax_lab.text(0.95, start + n / 2, TAG_LABELS[tag],
                    va="center", ha="right", fontsize=5)
    start += n

ax = fig.add_subplot(grid[0, 1])
sns.heatmap(ordered[value_cols], cmap="Blues", vmin=0, vmax=1, ax=ax,
            yticklabels=False, xticklabels=True,
            rasterized=True, cbar_kws={"label": "Fraction of sensory input"})
for b in np.cumsum(sizes.to_numpy())[:-1]:
    ax.hlines(y=b, xmin=0, xmax=len(value_cols),
              colors="red", linestyles="--", linewidth=0.5)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=5, rotation=-50, ha="left")
ax.set_ylabel("")
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=5)
cbar.set_label("Fraction of sensory input", fontsize=5)

fig.savefig("figure5d_postsynaptic_sensory_heatmap.svg", format="svg",
            bbox_inches="tight", dpi=600)
plt.show()


In [ ]:
# ---- panel e: pooled per population
pooled = postsyn.groupby("tag", observed=True)[COLUMN_ORDER].sum().reindex(TAGS)
pooled.columns = [DISPLAY.get(c, c) for c in COLUMN_ORDER]
pct = pooled.div(pooled.sum(axis=1), axis=0)

fig, axes = plt.subplots(nrows=len(TAGS), ncols=1, figsize=(6 * CM, 3 * CM), sharex=True)
fig.subplots_adjust(left=0.42, right=0.995, top=0.98, bottom=0.22, hspace=0.28)

for i, (ax, tag) in enumerate(zip(axes, TAGS)):
    row = pct.loc[tag].sort_values(ascending=False)
    left = 0.0
    for label, value in row.items():
        if value <= 0:
            continue
        # No edge colour: segments butt directly together, as in the published panel.
        # A white stroke would both insert gaps and completely erase the smallest
        # segments (some are ~0.1%, thinner than a 0.5 pt line).
        ax.barh(0, value * 100, left=left, height=0.62,
                color=SENSORY_COLOURS.get(label, "#cccccc"),
                edgecolor="none", linewidth=0)
        left += value * 100
    ax.set_xlim(0, 100)
    ax.set_yticks([])
    ax.annotate(TAG_LABELS[tag], xy=(0.0, 0.5), xycoords="axes fraction",
                xytext=(-6, 0), textcoords="offset points",
                ha="right", va="center", fontsize=5, clip_on=False)
    if i < len(axes) - 1:
        ax.get_xaxis().set_visible(False)
        for sp in ax.spines.values():
            sp.set_visible(False)
    else:
        ax.set_xticks([0, 25, 50, 75, 100])
        ax.set_xlabel("Sensory input fraction (%)", fontsize=5)
        for sp in ("top", "right", "left"):
            ax.spines[sp].set_visible(False)

fig.savefig("figure5e_postsynaptic_sensory_bars.svg", format="svg")
plt.show()

print((pct * 100).round(1).to_string())
